# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access and print dataset metadata (do not subscript like a dict)
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's enumerate all `recordSet` entries, and for each, the available field `@id`s, using only their `@id` values as references.

In [ ]:
# List all record sets available in the Croissant schema
record_sets = dataset.metadata.record_set
if not record_sets:
    print("No record sets found in metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet '@id': {rs['@id']}")
        if 'field' in rs:
            fields = rs['field']
            if isinstance(fields, dict):
                fields = [fields]
            print("  Fields:")
            for field in fields:
                print(f"   - Field '@id': {field['@id']} | Name: {field.get('name', 'N/A')}")
        else:
            print("  No fields found in this record set.")

## 3. Data Extraction
Load data from specific record set(s) into DataFrame(s) for analysis. Use only the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, the main data record set is likely to have '@id' = 'cr:RecordSet:survey'
# However, since the recordSets were printed above, set your value(s) here:
main_record_set_id = 'cr:RecordSet:survey'  # Update this value with the correct '@id' from overview if different

record_sets_ids = [main_record_set_id]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set {record_set_id}: {len(df)} records")

# Show columns of the main record set DataFrame
print(f"Fields/columns in DataFrame (from record set '@id' = {main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter records, normalize a numeric field, and group by a categorical field.

*Use only field `@id`s as references. Example field `@id`: 'cr:Field:phq9_total' and group-by field `@id`: 'cr:Field:gender'.*

In [ ]:
# Choose a numeric field (update if others available)
numeric_field = 'cr:Field:phq9_total'  # Typically, total score field for PHQ-9 assessment
group_field = 'cr:Field:gender'       # Group by gender

df = dataframes[main_record_set_id]

# Remove rows with missing values in the numeric field
filtered_df = df.dropna(subset=[numeric_field])

threshold = 10
filtered_df = filtered_df[filtered_df[numeric_field].astype(float) > threshold]
print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} records found")
print(filtered_df.head())

# Normalize the numeric field
normalized_col = f"{numeric_field}_normalized"
filtered_df[normalized_col] = (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) / filtered_df[numeric_field].astype(float).std()
print(f"Normalized values for {numeric_field} (first five):")
print(filtered_df[[numeric_field, normalized_col]].head())

# Group by a categorical variable, e.g. gender, and get mean by group
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Mean statistics grouped by {group_field}:\n")
    print(grouped_df[[numeric_field, normalized_col]].head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Import plotting library
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the PHQ-9 total scores
sns.histplot(df[numeric_field].astype(float).dropna(), bins=15, kde=True)
plt.title("Distribution of PHQ-9 Total Scores (cr:Field:phq9_total)")
plt.xlabel("PHQ-9 Total Score")
plt.ylabel("Frequency")
plt.show()

# If gender available, boxplot by gender
if group_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[group_field], y=df[numeric_field].astype(float))
    plt.title("PHQ-9 Total Scores by Gender")
    plt.xlabel("Gender")
    plt.ylabel("PHQ-9 Total Score")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides mental health survey records collected in Kilifi County, Kenya.
- Main assessment scores (e.g., PHQ-9) can be analyzed and grouped by demographic factors such as gender (using field `@id` references).
- The distribution and summary statistics can aid public health and local data science projects.
- Further domain-specific insights can be gained by exploring additional assessment instruments (e.g., GAD-7, PSQ) and categorical breakdowns.